In [1]:
import json
import time
import traceback
import inspect
import uuid
from litellm import completion
from dataclasses import dataclass, field
from typing import get_type_hints, List, Callable, Dict, Any, Optional

In [2]:
tools = {}
tools_by_tag = {}

def get_tool_metadata(func, tool_name=None, description=None, parameters_override=None, terminal=False, tags=None):
    tool_name = tool_name or func.__name__
    description = description or (func.__doc__.strip() if func.__doc__ else "No description provided.")

    if parameters_override is None:
        signature = inspect.signature(func)
        type_hints = get_type_hints(func)
        args_schema = {"type": "object", "properties": {}, "required": []}

        for param_name, param in signature.parameters.items():
            if param_name in ["action_context", "action_agent"]:
                continue

            def get_json_type(param_type):
                mapping = {str: "string", int: "integer", float: "number", bool: "boolean", list: "array", dict: "object"}
                return mapping.get(param_type, "string")

            param_type = type_hints.get(param_name, str)
            args_schema["properties"][param_name] = {"type": get_json_type(param_type)}

            if param.default == inspect.Parameter.empty:
                args_schema["required"].append(param_name)
    else:
        args_schema = parameters_override

    return {
        "tool_name": tool_name,
        "description": description,
        "parameters": args_schema,
        "function": func,
        "terminal": terminal,
        "tags": tags or []
    }


def register_tool(tool_name=None, description=None, parameters_override=None, terminal=False, tags=None):
    def decorator(func):
        metadata = get_tool_metadata(func, tool_name, description, parameters_override, terminal, tags)
        tools[metadata["tool_name"]] = {
            "description": metadata["description"],
            "parameters": metadata["parameters"],
            "function": metadata["function"],
            "terminal": metadata["terminal"],
            "tags": metadata["tags"] or []
        }
        for tag in metadata["tags"]:
            tools_by_tag.setdefault(tag, []).append(metadata["tool_name"])
        return func
    return decorator

In [3]:
@dataclass
class Prompt:
    messages: List[Dict] = field(default_factory=list)
    tools: List[Dict] = field(default_factory=list)
    metadata: dict = field(default_factory=dict)


class ActionContext:
    """Comparte recursos (como el propio LLM) entre el agente y sus tools."""
    def __init__(self, properties: dict = None):
        self.context_id = str(uuid.uuid4())
        self.properties = properties or {}

    def get(self, key, default=None):
        return self.properties.get(key, default)

    def set(self, key, value):
        self.properties[key] = value

In [4]:
def generate_response(prompt: Prompt) -> str:
    """Call LLM to get response"""
    messages = prompt.messages
    tools_param = prompt.tools

    if not tools_param:
        response = completion(
            model="groq/openai/gpt-oss-120b",
            messages=messages,
            max_tokens=1024,
            reasoning_effort="low"
        )
        return response.choices[0].message.content

    response = completion(
        model="groq/openai/gpt-oss-120b",
        messages=messages,
        tools=tools_param,
        max_tokens=1024,
        reasoning_effort="low"
    )

    message = response.choices[0].message
    if getattr(message, "tool_calls", None):
        tool_call = message.tool_calls[0]
        return json.dumps({
            "tool": tool_call.function.name,
            "args": json.loads(tool_call.function.arguments)
        })
    return json.dumps({"tool": "terminate", "args": {"message": message.content or ""}})

In [5]:
@dataclass(frozen=True)
class Goal:
    priority: int = 1
    name: str = ""
    description: str = ""


class Action:
    def __init__(self, name, function, description, parameters, terminal=False):
        self.name = name
        self.function = function
        self.description = description
        self.terminal = terminal
        self.parameters = parameters

    def execute(self, action_context: 'ActionContext' = None, **args) -> Any:
        sig_params = inspect.signature(self.function).parameters
        if "action_context" in sig_params:
            return self.function(action_context=action_context, **args)
        return self.function(**args)


class ActionRegistry:
    def __init__(self):
        self.actions = {}

    def register(self, action: Action):
        self.actions[action.name] = action

    def get_action(self, name: str) -> Optional[Action]:
        return self.actions.get(name, None)

    def get_actions(self) -> List[Action]:
        return list(self.actions.values())


class PythonActionRegistry(ActionRegistry):
    def __init__(self, tags: List[str] = None, tool_names: List[str] = None):
        super().__init__()
        self.terminate_tool = None

        for tool_name, tool_desc in tools.items():
            if tool_name == "terminate":
                self.terminate_tool = tool_desc

            if tool_names and tool_name not in tool_names:
                continue
            tool_tags = tool_desc.get("tags", [])
            if tags and not any(tag in tool_tags for tag in tags):
                continue

            self.register(Action(
                name=tool_name,
                function=tool_desc["function"],
                description=tool_desc["description"],
                parameters=tool_desc.get("parameters", {}),
                terminal=tool_desc.get("terminal", False)
            ))

        # Aseguramos que terminate SIEMPRE esté disponible, sin importar los tags pedidos
        if self.terminate_tool and "terminate" not in self.actions:
            self.register(Action(
                name="terminate",
                function=self.terminate_tool["function"],
                description=self.terminate_tool["description"],
                parameters=self.terminate_tool.get("parameters", {}),
                terminal=True
            ))

In [6]:
class Memory:
    def __init__(self):
        self.items = []

    def add_memory(self, memory: dict):
        self.items.append(memory)

    def get_memories(self, limit: int = None) -> List[Dict]:
        return self.items[:limit]

    def copy_without_system_memories(self):
        filtered = [m for m in self.items if m["type"] != "system"]
        memory = Memory()
        memory.items = filtered
        return memory


class Environment:
    def execute_action(self, action_context: ActionContext, action: Action, args: dict) -> dict:
        try:
            result = action.execute(action_context=action_context, **args)
            return self.format_result(result)
        except Exception as e:
            return {"tool_executed": False, "error": str(e), "traceback": traceback.format_exc()}

    def format_result(self, result: Any) -> dict:
        return {"tool_executed": True, "result": result, "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z")}


class PythonEnvironment(Environment):
    """Mismo comportamiento que Environment; existe por convención de nombres del curso."""
    pass

In [7]:
class AgentLanguage:
    def construct_prompt(self, actions, environment, goals, memory) -> Prompt:
        raise NotImplementedError()

    def parse_response(self, response: str) -> dict:
        raise NotImplementedError()


class AgentFunctionCallingActionLanguage(AgentLanguage):
    def format_goals(self, goals: List[Goal]) -> List[Dict]:
        sep = "\n-------------------\n"
        goal_text = "\n\n".join([f"{g.name}:{sep}{g.description}{sep}" for g in sorted(goals, key=lambda x: x.priority)])
        return [{"role": "system", "content": goal_text}]

    def format_memory(self, memory: Memory) -> List[Dict]:
        mapped = []
        for item in memory.get_memories():
            content = item.get("content") or json.dumps(item, indent=4)
            role = "assistant" if item["type"] in ("assistant", "environment") else "user"
            mapped.append({"role": role, "content": content})
        return mapped

    def format_actions(self, actions: List[Action]) -> List[Dict]:
        return [
            {
                "type": "function",
                "function": {
                    "name": a.name,
                    "description": a.description[:1024],
                    "parameters": a.parameters or {"type": "object", "properties": {}},
                },
            } for a in actions
        ]

    def construct_prompt(self, actions, environment, goals, memory) -> Prompt:
        messages = self.format_goals(goals) + self.format_memory(memory)
        tools_list = self.format_actions(actions)
        return Prompt(messages=messages, tools=tools_list)

    def parse_response(self, response: str) -> dict:
        try:
            return json.loads(response)
        except Exception:
            return {"tool": "terminate", "args": {"message": response}}

In [8]:
class Agent:
    def __init__(self, goals, agent_language, action_registry, generate_response, environment):
        self.goals = goals
        self.generate_response = generate_response
        self.agent_language = agent_language
        self.actions = action_registry
        self.environment = environment
        self.action_context = ActionContext({"llm": generate_response})

    def construct_prompt(self, goals, memory, actions) -> Prompt:
        return self.agent_language.construct_prompt(actions.get_actions(), self.environment, goals, memory)

    def get_action(self, response):
        invocation = self.agent_language.parse_response(response)
        action = self.actions.get_action(invocation["tool"])
        return action, invocation

    def should_terminate(self, response: str) -> bool:
        action_def, _ = self.get_action(response)
        return action_def.terminal

    def set_current_task(self, memory, task):
        memory.add_memory({"type": "user", "content": task})

    def update_memory(self, memory, response, result):
        memory.add_memory({"type": "assistant", "content": response})
        memory.add_memory({"type": "environment", "content": json.dumps(result)})

    def prompt_llm_for_action(self, full_prompt: Prompt) -> str:
        return self.generate_response(full_prompt)

    def run(self, user_input: str, memory=None, max_iterations: int = 50) -> Memory:
        memory = memory or Memory()
        self.set_current_task(memory, user_input)

        for _ in range(max_iterations):
            prompt = self.construct_prompt(self.goals, memory, self.actions)
            print("Agent thinking...")
            response = self.prompt_llm_for_action(prompt)
            print(f"Agent Decision: {response}")

            action, invocation = self.get_action(response)
            result = self.environment.execute_action(self.action_context, action, invocation["args"])
            print(f"Action Result: {result}")

            self.update_memory(memory, response, result)
            if self.should_terminate(response):
                break

        return memory

In [9]:
@register_tool(tags=["system"], terminal=True)
def terminate(message: str) -> str:
    """Termina la ejecución del agente con un mensaje final.

    Args:
        message: El mensaje final antes de terminar

    Returns:
        El mensaje con una nota de terminación
    """
    return f"{message}\nTerminando..."